In [ ]:
import numpy as np
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import random
import dask.dataframe as dd
import sys
sys.path.append('../utilities')
import utilities as utils
import pandas as pd

In [ ]:

# read in LF simulations
version = 'n1.4'
if not os.path.exists(f'out/{version}'):
   os.makedirs(f'out/{version}')

filelist = utils.get_all_files(f"../simulation/out/LF/{version}/tier2/",".csv")
x = np.empty([0,12])
for file in filelist:
   print(file)
   #df = pd.read_csv(file)
   #del df['nprimaries']
   #df.insert(len(df.columns),'nprimaries',np.ones(len(df)))
   #print(df)
   #df.to_csv(file)
   df = dd.read_csv(file)
   x = np.append(x,[np.array([0., 
                              df["radius"].compute()[0], 
                              df["thickness"].compute()[0], 
                              df["npanels"].compute()[0], 
                              df["theta"].compute()[0], df["length"].compute()[0], 
                              np.sum(df["total_nC_Ge77[cts]"].compute()), 
                              np.sum(df["prod_rate_Ge77[nuc/(kg*yr)]"].compute()),
                              df["nC_Ge77_scaling"].compute()[0], np.sum(df["nprimaries"].compute()),
                              np.sum(df["nprimaries"].compute())+np.sum(df["nsec"].compute()), 
                              np.sum(df["nsec"].compute())])],
                              axis=0)

nLF=len(filelist)
# read in HF simulations
version = 'v1.1'
if not os.path.exists(f'out/{version}'):
   os.makedirs(f'out/{version}')
nfiles = [202, 185, 193, 197]
filelist = utils.get_all_files(f"../simulation/out/HF/{version}/tier2/neutron-sim-HF-n1.1",".csv")
print(filelist)
for i, file in enumerate(filelist):
   df = dd.read_csv(file)
   x = np.append(x,[np.array([1., df["radius"].compute()[0], df["thickness"].compute()[0], df["npanels"].compute()[0], df["theta"].compute()[0], df["length"].compute()[0], np.sum(df["total_nC_Ge77[cts]"].compute()), np.sum(df["prod_rate_Ge77[nuc/(kg*yr)]"].compute()),df["nC_Ge77_scaling"].compute()[0],np.sum(df["nprimaries"].compute()),len(df.compute())+np.sum(df["nsec"].compute()),np.sum(df["nsec"].compute())])],axis=0)

df_new = dd.from_array(x, columns = ["mode","radius","thickness","npanels","theta","length", "nC_Ge77[cts]", "production", "scaling", "nprimaries","neutrons","nsec"])
nHF = len(filelist)
version = 'n1.4'
print(df_new.compute())

In [ ]:
r_water = 11./2.
frac_enter_exp = 0.27
rate_muon = 3.6 * 1.e-4 * (np.pi*r_water**2) * (60.*60.*24.*365)
frac_neutrons_LAr = np.zeros(nHF+nLF)
frac_neutrons_LAr[:nLF] = 1./0.13
frac_neutrons_LAr[nLF:] = 1.

factor = (df_new["nprimaries"].to_dask_array().compute()+df_new["nsec"].to_dask_array().compute()) * frac_enter_exp * frac_neutrons_LAr / rate_muon
print(df_new["nprimaries"].to_dask_array().compute()[0]+df_new["nsec"].to_dask_array().compute()[0],df_new["nC_Ge77[cts]"].to_dask_array().compute()[0],rate_muon)

In [ ]:
rate_Ge77 = df_new["nC_Ge77[cts]"].to_dask_array().compute()/factor/1000.
df_new["rate_Ge77[nuc/kg/yr]"]=dd.from_array(np.round(rate_Ge77,5))
print(rate_Ge77)

In [ ]:
fig, _ = plt.subplots(3,2,sharey=True,figsize=(12, 12) ,layout="constrained")
ax = fig.axes
ax[0].plot(df_new["radius"].to_dask_array().compute()[:-6],rate_Ge77[:-6], "o",color="teal", label="low fidelity")
ax[0].plot(df_new["radius"].to_dask_array().compute()[-6:],rate_Ge77[-6:], "o",color="orangered", label="high fidelity")
ax[0].set_xlabel("radius [cm]", fontsize=10)
ax[0].set_ylabel(r"$^{77}Ge$ production rate [nuc/(kg$\cdot$yr)]", fontsize=10)
ax[0].legend(loc="lower left")

ax[1].plot(df_new["thickness"].to_dask_array().compute()[:-6],rate_Ge77[:-6], "o",color="teal", label="low fidelity")
ax[1].plot(df_new["thickness"].to_dask_array().compute()[-6:],rate_Ge77[-6:], "o",color="orangered", label="high fidelity")
ax[1].set_xlabel("thickness [cm]", fontsize=10)

fig = plt.figure(figsize=(8,6))
ax[2].plot(df_new["npanels"].to_dask_array().compute()[:-6],rate_Ge77[:-6], "o",color="teal", label="low fidelity")
ax[2].plot(df_new["npanels"].to_dask_array().compute()[-6:],rate_Ge77[-6:], "o",color="orangered", label="high fidelity")
ax[2].set_xlabel("number of panels", fontsize=10)
ax[2].set_ylabel(r"$^{77}Ge$ production rate [nuc/(kg$\cdot$yr)]", fontsize=10)

ax[3].plot(df_new["theta"].to_dask_array().compute()[:-6],rate_Ge77[:-6], "o",color="teal", label="low fidelity")
ax[3].plot(df_new["theta"].to_dask_array().compute()[-6:],rate_Ge77[-6:], "o",color="orangered", label="high fidelity")
ax[3].set_xlabel("angle of panels [deg]", fontsize=10)


ax[4].plot(df_new["length"].to_dask_array().compute()[:-6],rate_Ge77[:-6], "o",color="teal", label="low fidelity")
ax[4].plot(df_new["length"].to_dask_array().compute()[-6:],rate_Ge77[-6:], "o",color="orangered", label="high fidelity")
ax[4].set_xlabel("length [cm]", fontsize=10)
ax[4].set_ylabel(r"$^{77}Ge$ production rate [nuc/(kg$\cdot$yr)]", fontsize=10)
ax[5].set_axis_off()


In [ ]:
def monte_carlo_simulations(rate_mean, rate_std, n_iter=10000):
    outputs = []
    for i in range(n_iter):
        result = random.normalvariate(rate_mean, rate_std)
        outputs.append(result)
    return outputs

def gauss(x, A, mean, sigma):
    return A * np.exp(-(x - mean) ** 2 / (2 * sigma ** 2))

def Ge77MonteCarlosSim(nevt,factor):
    sigma = []
    for i,evt in enumerate(nevt):
        nevt_std=np.sqrt(evt)
        events = monte_carlo_simulations(evt,nevt_std)
        events = np.array(events) * factor[i]
        values, bins = np.histogram(events, bins=100)    
        x = (bins[:-1] + bins[1:]) / 2
        plt.scatter(x,values)
        parameters, covariance = curve_fit(gauss, x, values, p0=[250, evt*factor[i], nevt_std*factor[i]])
        plt.plot(x,gauss(x,parameters[0],parameters[1],parameters[2]))
        sigma.append(parameters[2])
    return sigma

In [ ]:
sigma = Ge77MonteCarlosSim(df_new["nC_Ge77[cts]"].to_dask_array().compute()[:nLF],1./(factor[:nLF]*1000.))


In [ ]:
plt.figure(figsize=(10,4))
mean = np.mean(sigma)
std = np.std(sigma)
plt.axhline(y = mean, color = 'orangered')
x=[i for i in range(0,265)]
plt.fill_between(x,mean-std,mean+std,color='coral', alpha=0.3)
plt.plot(df_new["radius"].to_dask_array().compute()[:nLF],sigma,"o",color="teal")
plt.xlabel('Radius [cm]')
plt.ylabel(r'$\sigma\,\rm [nuclei/(kg \cdot yr)]$')
plt.text(210,0.012,f"mean = {np.round(mean,4)} +- {np.round(std,4)}",fontsize=8)

plt.savefig(f"out/{version}/noise_LF_{version}.png")

In [ ]:
f=open(f'out/{version}/Ge77_rates_{version}.csv',"w")
f.write(f"# LF noise: mean = {mean:.5f} +- {std:.5f}"+"\n")
f.close()
df_new.to_csv(f'out/{version}/Ge77_rates_{version}.csv',mode='a')